# FBref Premier League match scraper (updated)
**Heads-up:** in Jan 2026 FBref lost its Opta licence and removed advanced stats (xG etc.). Historical pages may still have some tables, but the shooting table this notebook merges may be missing. The code below handles that gracefully (falls back to match results only). FBref also rate-limits to roughly 10 requests/minute, so requests are throttled.

In [1]:
import io
import time
from datetime import date

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
BASE = "https://fbref.com"
DELAY = 6.5  # seconds between requests (FBref allows ~10 requests/min)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
})

def fetch(url, retries=3):
    """GET a page politely: throttle, raise on HTTP errors, back off on 429."""
    for attempt in range(retries):
        time.sleep(DELAY)
        resp = session.get(url, timeout=30)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 60))
            print(f"Rate limited, sleeping {wait}s")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.text
    raise RuntimeError(f"Failed to fetch {url}")

def read_table(html, match):
    """pandas >= 2.1 deprecates passing literal HTML; wrap it in StringIO."""
    return pd.read_html(io.StringIO(html), match=match)[0]

## Explore one team first

In [3]:
standings_url = f"{BASE}/en/comps/9/Premier-League-Stats"
html = fetch(standings_url)

HTTPError: 403 Client Error: Forbidden for url: https://fbref.com/en/comps/9/Premier-League-Stats

In [ ]:
soup = BeautifulSoup(html, "lxml")
standings_table = soup.select("table.stats_table")[0]
links = [a.get("href") for a in standings_table.find_all("a")]
links = [l for l in links if l and "/squads/" in l]
team_urls = [f"{BASE}{l}" for l in dict.fromkeys(links)]  # de-dupe, keep order
team_urls[:3]

In [ ]:
team_html = fetch(team_urls[0])
matches = read_table(team_html, "Scores & Fixtures")
matches.head()

In [ ]:
def get_shooting(team_html):
    """Return the team's shooting table, or None if FBref no longer serves it."""
    soup = BeautifulSoup(team_html, "lxml")
    links = [a.get("href") for a in soup.find_all("a")]
    links = [l for l in links if l and "all_comps/shooting/" in l]
    if not links:
        return None
    try:
        shooting = read_table(fetch(f"{BASE}{links[0]}"), "Shooting")
    except (ValueError, requests.HTTPError):
        return None
    if isinstance(shooting.columns, pd.MultiIndex):
        shooting.columns = shooting.columns.droplevel()
    return shooting

SHOOT_COLS = ["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]

def build_team_data(matches, shooting):
    if shooting is None or not set(SHOOT_COLS).issubset(shooting.columns):
        print("  shooting stats unavailable, keeping match results only")
        return matches.copy()
    return matches.merge(shooting[SHOOT_COLS], on="Date")

In [ ]:
shooting = get_shooting(team_html)
team_data = build_team_data(matches, shooting)
team_data.head()

## Scrape multiple seasons

In [ ]:
# Season label = year the season ends (2022 -> 2021-22). Current season first.
today = date.today()
current = today.year + 1 if today.month >= 8 else today.year
N_SEASONS = 2
years = list(range(current, current - N_SEASONS, -1))
years

In [ ]:
all_matches = []
standings_url = f"{BASE}/en/comps/9/Premier-League-Stats"

for year in years:
    soup = BeautifulSoup(fetch(standings_url), "lxml")
    standings_table = soup.select("table.stats_table")[0]

    links = [a.get("href") for a in standings_table.find_all("a")]
    links = [l for l in links if l and "/squads/" in l]
    team_urls = [f"{BASE}{l}" for l in dict.fromkeys(links)]

    prev = soup.select("a.prev")
    standings_url = f"{BASE}{prev[0].get('href')}" if prev else None

    for team_url in team_urls:
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        print(year, team_name)
        try:
            team_html = fetch(team_url)
            matches = read_table(team_html, "Scores & Fixtures")
            team_data = build_team_data(matches, get_shooting(team_html))
        except (ValueError, requests.HTTPError) as e:
            print(f"  skipped: {e}")
            continue

        team_data = team_data[team_data["Comp"] == "Premier League"].copy()
        team_data["Season"] = year
        team_data["Team"] = team_name
        all_matches.append(team_data)

    if standings_url is None:
        break

In [ ]:
len(all_matches)

In [ ]:
match_df = pd.concat(all_matches, ignore_index=True)
match_df.columns = [c.lower() for c in match_df.columns]
match_df

In [ ]:
match_df.to_csv("matches.csv", index=False)